**Single-Issuer Credit Screener**
-
**Purpose:** Input any public company's ticker and pull its credit profile automatically:
- **Fundamentals & credit metrics** (fully automated): SEC EDGAR XBRL API — debt, EBITDA proxy,
  interest expense, cash, leverage, and interest coverage
- **Bond pricing/yield** (somewhat manual step): FINRA's Fixed Income Data Center doesn't expose
  a free public bulk search API, so this notebook walks you through looking up the issuer's
  bonds there and recording what you find — including a clean check for issuers with
  **no outstanding bonds before maturity**

Works for any public company, useful as a general credit due-diligence
tool.


## 0: Setup

In [ ]:
#Import libraries
import requests
import pandas as pd

# SEC REQUIRES a descriptive User-Agent identifying you, so replace with your real info.
HEADERS = {
    "User-Agent": "NAME - PROJECT NAME (your-email@example.com)"
}
SEC_BASE = "https://data.sec.gov"


## 1: Choose your issuer

Enter any ticker (e.g. `"AAPL"` for Apple Inc, `"GOOG"`, `"MSFT"`). This drives everything
downstream.


In [ ]:
TICKER = "XXXX"   #change to any ticker you want to analyze


## 2: Resolve ticker to CIK

EDGAR is keyed by CIK, not ticker. We download SEC's official mapping file once and look up
the CIK for your chosen ticker.


In [ ]:
#load CIK map from SEC
def load_ticker_cik_map():
    url = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    df = pd.DataFrame(list(resp.json().values()))
    df["cik_str"] = df["cik_str"].astype(str).str.zfill(10)
    return df.set_index("ticker")

ticker_cik_map = load_ticker_cik_map()

#Check for valid ticker
if TICKER not in ticker_cik_map.index:
    raise ValueError(f"Ticker '{TICKER}' not found in SEC's ticker list. "
                      f"Double check the ticker (SEC uses the primary listing symbol).")

#Resolve ticker to CIK and print
cik = ticker_cik_map.loc[TICKER, "cik_str"]
company_name = ticker_cik_map.loc[TICKER, "title"]
print(f"{TICKER} -> CIK {cik} ({company_name})")


## 3: Pull fundamentals from EDGAR

Fetches every XBRL fact this company has reported. We'll extract just what we need next.


In [ ]:
#Get facts from SEC
def get_company_facts(cik):
    url = f"{SEC_BASE}/api/xbrl/companyfacts/CIK{cik}.json"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

#resolve and print
facts = get_company_facts(cik)
print(f"Pulled {len(facts.get('facts', {}).get('us-gaap', {}))} us-gaap tags for {company_name}")


## 4: Extract credit metrics

XBRL tagging varies by company, so each metric has a few candidate tag names (most-preferred
first). If a metric comes back as `None` after running this, print
`facts['facts']['us-gaap'].keys()` to find the tag this specific company actually uses, and
add it to the relevant list below.


In [ ]:
#Get relevant metrics by searching by tag
METRIC_TAGS = {
    "total_debt": [
        "DebtLongtermAndShorttermCombinedAmount",
        "LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities",
        "LongTermDebtNoncurrent",
        "LongTermDebt",
    ],
    "operating_income": ["OperatingIncomeLoss"],
    "depreciation_amortization": [
        "DepreciationDepletionAndAmortization",
        "DepreciationAmortizationAndAccretionNet",
        "DepreciationAndAmortization",
    ],
    "interest_expense": [
        "InterestExpense",
        "InterestExpenseDebt",
        "InterestAndDebtExpense",
    ],
    "cash": ["CashAndCashEquivalentsAtCarryingValue"],
}


def get_latest_annual_value(facts, tag_candidates):
    us_gaap = facts.get("facts", {}).get("us-gaap", {})
    for tag in tag_candidates:
        if tag not in us_gaap:
            continue
        units = us_gaap[tag].get("units", {}).get("USD", [])
        annual_points = [p for p in units if p.get("form") == "10-K" and p.get("fp") == "FY"]
        if not annual_points:
            continue
        latest = max(annual_points, key=lambda p: p["end"])
        return latest["val"], latest["end"], tag
    return None, None, None



metrics = {"ticker": TICKER, "company_name": company_name}
for metric, tags in METRIC_TAGS.items():
    val, period_end, tag_used = get_latest_annual_value(facts, tags)
    metrics[metric] = val
    metrics[f"{metric}_period"] = period_end
    metrics[f"{metric}_tag_used"] = tag_used

#use dataframe for ease
metrics_df = pd.Series(metrics)
metrics_df


periods_found = {m: metrics[f"{m}_period"] for m in METRIC_TAGS if metrics[f"{m}_period"] is not None}
dist_per = set(periods_found.values())

if len(dist_per) > 1:
  print(f"Warning: Metrics are pulled from mismatched reporting periods. "
        "Ratios computed may have different fiscal years")

## 5: Compute leverage and coverage ratios

- **EBITDA proxy** = Operating Income + D&A
- **Leverage** = Total Debt / EBITDA
- **Interest Coverage** = EBITDA / Interest Expense


In [ ]:
ebitda_proxy = None
leverage = None
coverage = None

#calculate metrics
if metrics["operating_income"] is not None and metrics["depreciation_amortization"] is not None:
    ebitda_proxy = metrics["operating_income"] + metrics["depreciation_amortization"]

if ebitda_proxy and metrics["total_debt"] is not None:
    leverage = metrics["total_debt"] / ebitda_proxy

if ebitda_proxy and metrics["interest_expense"] is not None:
    coverage = ebitda_proxy / metrics["interest_expense"]

#print
print(f"{company_name} ({TICKER}) Credit Snapshot")
print(f"  Total Debt:          {metrics['total_debt']:,.0f}" if metrics['total_debt'] else "  Total Debt:          N/A")
print(f"  EBITDA (proxy):      {ebitda_proxy:,.0f}" if ebitda_proxy else "  EBITDA (proxy):      N/A")
print(f"  Interest Expense:    {metrics['interest_expense']:,.0f}" if metrics['interest_expense'] else "  Interest Expense:    N/A")
print(f"  Cash:                {metrics['cash']:,.0f}" if metrics['cash'] else "  Cash:                N/A")
print(f"  Leverage (Debt/EBITDA):     {leverage:.2f}x" if leverage else "  Leverage (Debt/EBITDA):     N/A")
print(f"  Interest Coverage (EBITDA/Int): {coverage:.2f}x" if coverage else "  Interest Coverage (EBITDA/Int): N/A")


## 6: Bond info entry

Look up one bond for this issuer on FINRA's Fixed Income Data Center

**If no bonds appear at all**, set `bond_found = False` -- the cell will flag that this issuer
has no outstanding public bond debt

If a bond is found, fill in its terms exactly as FINRA displays them: market price and call
prices are quoted **per 100 of face value** (e.g. a price of 95.25 means $952.50 per $1,000
bond -- standard bond quoting convention).

**Call schedule:** many corporate bonds are callable on more than one date, typically at a
step-down premium (e.g. callable at 103 in year 3, 101.5 in year 4, 100 in year 5+). List every
call date/price you can find in the bond's prospectus or FINRA's listing as a separate entry in
`call_schedule` -- Step 7 will compute YTC for each one and use the worst (lowest) as part of
yield-to-worst.


In [ ]:
#bond existence check
bond_found = True  # set to False if no bonds turned up on FINRA for this issuer

#input bond details HERE
bond_inputs = {
    "cusip": "XXXXXXXXX",
    "face_value": 1000,        # bond denomination, typically $1000
    "coupon_rate": X.XX,        # annual coupon, in percent
    "coupon_freq": X,          # payments per year
    "market_price": XX.XX,     # price per 100 of face value, as quoted on FINRA
    "maturity_date": "YYYY-MM-DD",
}

# One entry per call date. Leave list empty if bond not callable.
call_schedule = [
    {"date": "YYYY-MM-DD", "price": XXX.XX},
    {"date": "YYYY-MM-DD", "price": XXX.XX},
    {"date": "YYYY-MM-DD", "price": XXX.XX},
]

#If bond doesn't exist
if not bond_found:
    print("No outstanding bonds recorded for this issuer.")
    print("This may mean: (a) no public bond debt outstanding, (b) debt sits at a different "
          "legal entity/subsidiary (check parent/sub structure), or (c) you haven't looked it "
          "up yet -- see the FINRA lookup instructions above.")


## 7: Yield calculations (YTM, YTC, YTW)

Using the bond terms and call schedule from Step 6, computes:
- **Current yield** -- annual coupon / market price
- **Yield to Maturity (YTM)** -- solved via bisection on the bond pricing equation
- **Yield to Call (YTC)** -- computed separately for *every* date in `call_schedule`
- **Yield to Worst (YTW)** -- the minimum across YTM and every YTC.


In [ ]:
#get data
from datetime import date

#functions
def bond_price_from_yield(coupon_per_period, periods, annual_yield, freq, redemption_value):
    """Present value of a bond's cash flows given an annual yield (bond-equivalent, i.e.
    periodic rate = annual_yield / freq -- the standard US market convention)."""
    r = annual_yield / freq
    price = sum(coupon_per_period / (1 + r) ** t for t in range(1, periods + 1))
    price += redemption_value / (1 + r) ** periods
    return price


def solve_yield(target_price, coupon_per_period, periods, redemption_value, freq,
                 low=0.0001, high=2.0, tol=1e-7, max_iter=200):
    """Bisection solve for the annual yield that prices a bond at target_price.
    Bond price is monotonically decreasing in yield, so bisection is safe and simple --
    no need for scipy or an initial guess."""
    for _ in range(max_iter):
        mid = (low + high) / 2
        p = bond_price_from_yield(coupon_per_period, periods, mid, freq, redemption_value)
        if abs(p - target_price) < tol:
            return mid
        if p > target_price:
            low = mid
        else:
            high = mid
    return mid


def years_between(start, end):
    return (end - start).days / 365.25

#initialize variables
ytm = ytw = current_yield = None
ytw_years = None   # years to the horizon (maturity or the worst call date) that produced YTW
ytc_results = []    # list of (call_date_str, years_to_call, ytc) for every call date

#run calculations and return
if bond_found:
    today = date.today()
    freq = bond_inputs["coupon_freq"]
    face = bond_inputs["face_value"]
    coupon_per_period = face * (bond_inputs["coupon_rate"] / 100) / freq
    market_price_dollars = bond_inputs["market_price"] / 100 * face

    # --- Yield to Maturity ---
    maturity_date = date.fromisoformat(bond_inputs["maturity_date"])
    years_to_maturity = years_between(today, maturity_date)
    periods_to_maturity = max(round(years_to_maturity * freq), 1)
    ytm = solve_yield(market_price_dollars, coupon_per_period, periods_to_maturity, face, freq)

    # --- Yield to Call, for every date in the call schedule ---
    for call in call_schedule:
        call_date = date.fromisoformat(call["date"])
        years_to_call = years_between(today, call_date)
        if years_to_call <= 0:
            continue  # skip call dates already in the past
        periods_to_call = max(round(years_to_call * freq), 1)
        call_redemption = call["price"] / 100 * face
        ytc = solve_yield(market_price_dollars, coupon_per_period, periods_to_call, call_redemption, freq)
        ytc_results.append((call["date"], years_to_call, ytc))

    # --- Current yield ---
    annual_coupon_dollars = face * (bond_inputs["coupon_rate"] / 100)
    current_yield = annual_coupon_dollars / market_price_dollars

    # --- Yield to worst: minimum across YTM and every YTC ---
    candidates = [("Maturity", years_to_maturity, ytm)] + \
                 [(f"Call {d}", y, r) for d, y, r in ytc_results]
    worst_label, ytw_years, ytw = min(candidates, key=lambda c: c[2])

    print(f"Bond: {bond_inputs['cusip']}  |  Coupon {bond_inputs['coupon_rate']}%  |  "
          f"Matures {bond_inputs['maturity_date']}")
    print(f"  Market Price:      {bond_inputs['market_price']:.3f}  (per 100 face)")
    print(f"  Current Yield:     {current_yield:.2%}")
    print(f"  Yield to Maturity: {ytm:.2%}   ({years_to_maturity:.1f} yrs to maturity)")
    for d, y, r in ytc_results:
        print(f"  Yield to Call {d}: {r:.2%}   ({y:.1f} yrs)")
    print(f"  Yield to Worst:    {ytw:.2%}   <-- worst case is '{worst_label}' -- use this number")
else:
    print("No bond entered in Step 6 -- nothing to calculate.")


## 8: Comparison / conclusion

Instead of comparing YTW to an arbitrary flat threshold, this pulls the live U.S. Treasury
par yield curve and interpolates
it to match the bond's yield-to-worst horizon, so the comparison is a proper credit spread
(YTW minus the risk-free rate for a similar maturity) rather than a fixed number.

**Note:** If the fetch fails or returns nothing, fill in
`manual_treasury_yield` below with a rate you look up directly at
https://home.treasury.gov/resource-center/data-chart-center/interest-rates (pick the tenor
closest to the bond's YTW horizon) and the rest of the analysis still runs.


In [ ]:
import re
from datetime import date as _date

# e.g. 0.0425 for 4.25% -- fill in if the auto-fetch below fails
manual_treasury_yield = None


TREASURY_TENORS_YEARS = {
    "1MONTH": 1/12, "2MONTH": 2/12, "3MONTH": 3/12, "4MONTH": 4/12, "6MONTH": 6/12,
    "1YEAR": 1, "2YEAR": 2, "3YEAR": 3, "5YEAR": 5, "7YEAR": 7,
    "10YEAR": 10, "20YEAR": 20, "30YEAR": 30,
}

#pull from treasury function
def fetch_treasury_curve(yyyymm=None):
    """Pull the most recent day's Treasury par yield curve for a given month from
    Treasury.gov's public XML feed. Returns {tenor_years: rate_decimal}."""
    if yyyymm is None:
        yyyymm = _date.today().strftime("%Y%m")
    url = (f"https://home.treasury.gov/resource-center/data-chart-center/interest-rates/"
           f"pages/xmlview?data=daily_treasury_yield_curve&field_tdr_date_value_month={yyyymm}")
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    entries = re.findall(r"<entry.*?>.*?</entry>", resp.text, re.DOTALL)
    if not entries:
        # Current month may not have data posted yet (e.g. very early in the month) --
        # fall back to the previous month automatically
        y, m = int(yyyymm[:4]), int(yyyymm[4:])
        prev_yyyymm = f"{y-1}12" if m == 1 else f"{y}{m-1:02d}"
        if prev_yyyymm != yyyymm:
            prev_url = (f"https://home.treasury.gov/resource-center/data-chart-center/"
                        f"interest-rates/pages/xmlview?data=daily_treasury_yield_curve&"
                        f"field_tdr_date_value_month={prev_yyyymm}")
            resp = requests.get(prev_url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            entries = re.findall(r"<entry.*?>.*?</entry>", resp.text, re.DOTALL)
        if not entries:
            raise ValueError(f"No data returned for {yyyymm} or {prev_yyyymm}.")
    latest = entries[-1]  # most recent trading day published that month
    curve = {}
    for tenor_code, years in TREASURY_TENORS_YEARS.items():
        m = re.search(rf"<d:BC_{tenor_code}[^>]*>([\d.]*)</d:BC_{tenor_code}>", latest)
        if m and m.group(1):
            curve[years] = float(m.group(1)) / 100
    return curve


#interpolate to bond lifespan
def interpolate_treasury_yield(curve, target_years):
    """Linearly interpolate the par curve to the target maturity (in years)."""
    tenors = sorted(curve.keys())
    if target_years <= tenors[0]:
        return curve[tenors[0]]
    if target_years >= tenors[-1]:
        return curve[tenors[-1]]
    for t0, t1 in zip(tenors, tenors[1:]):
        if t0 <= target_years <= t1:
            frac = (target_years - t0) / (t1 - t0)
            return curve[t0] + frac * (curve[t1] - curve[t0])


#return and default to manual entry if fetch fails
treasury_yield = None
if bond_found and ytw is not None:
    try:
        curve = fetch_treasury_curve()
        treasury_yield = interpolate_treasury_yield(curve, ytw_years)
        print(f"Treasury curve fetched -- interpolated {ytw_years:.1f}yr rate: {treasury_yield:.2%}")
    except Exception as e:
        print(f"Auto-fetch failed ({e}) -- using manual_treasury_yield if set.")
        treasury_yield = manual_treasury_yield
        if treasury_yield is not None:
            print(f"Using manual entry: {treasury_yield:.2%}")

### Conclusion

General credit read combining leverage, coverage, and the bond's spread over Treasuries --
applicable to any issuer/bond pair, not specific to any particular use case.


In [ ]:
if bond_found and leverage is not None and coverage is not None and ytw is not None:
    # Leverage/coverage tiers -- adjust thresholds as you develop your own credit views
    if leverage < 3.7:
        leverage_tier = "Low"
    elif leverage < 7:
        leverage_tier = "Moderate"
    else:
        leverage_tier = "High"

    if coverage > 4:
        coverage_tier = "Strong"
    elif coverage > 2:
        coverage_tier = "Moderate"
    else:
        coverage_tier = "Weak"

    print(f"{company_name} ({TICKER}) -- Credit Summary")
    print(f"  Leverage (Debt/EBITDA):   {leverage:.2f}x   [{leverage_tier}]")
    print(f"  Interest Coverage:        {coverage:.2f}x   [{coverage_tier}]")
    print(f"  Bond Yield to Worst:      {ytw:.2%}   (~{ytw_years:.1f}yr horizon)")

    spread = None
    if treasury_yield is not None:
        spread = ytw - treasury_yield
        spread_bps = spread * 10000
        print(f"  Matched Treasury Yield:   {treasury_yield:.2%}")
        print(f"  Spread over Treasuries:   {spread_bps:+.0f} bps")
    else:
        print("  Spread over Treasuries:   N/A (no Treasury benchmark available)")
    print()

    exceptional = leverage < 1.5 and coverage > 15

    # Simple, transparent rule-based read -- a starting lens, not a substitute for judgment
    if spread is not None:
        if exceptional and spread * 10000 > 200:
            read = ("Wide spread despite quite strong fundamentals, highly unusual")
        elif spread * 10000 > 600 and leverage_tier == "High" and coverage_tier == "Weak":
            read = ("Wide spread and weak fundamentals are generally associated with high "
                    "risk credit, especially those pointing to refinancing/covenant terms.")
        elif spread * 10000 > 600 and leverage_tier in ("Low", "Moderate") and coverage_tier in ("Strong", "Moderate"):
            read = ("Wide spread despite reasonable leverage/coverage indicates there may "
                    "be factors beyond the financials impacting price, needs further research "
                    "but there is some potential.")
        elif spread * 10000 < 200:
            read = ("Tight spread relative to Treasuries indicating the yield offers limited compensation "
                    "over the risk-free rate given the credit risk involved.")
        else:
            read = "Moderate spread -- roughly in line with what the leverage/coverage profile would suggest."
    else:
        read = "Add a Treasury benchmark (auto-fetch or manual) to get a spread-based read."

    print("Read:", read)
else:
    print("Need Step 5 financials and a Step 6/7 bond entry to run this comparison.")


## Summary

Change `TICKER` in Step 1 and `bond_inputs` in Step 6 to analyze a different issuer/bond, then
re-run top to bottom. Steps 2-5 (fundamentals) are fully automated for any public company;
Step 6 (bond terms) requires a quick manual lookup on FINRA; Steps 7-8 compute yields and tie
everything together into a quick read.

Run this as a sanity check or preliminary read
